# C12-classical-models — Practice p08 — Solution


Activity is determined from signed margins. The intercept is not regularized, and equality contributes zero by the strict mask.


In [ ]:
import numpy as np

X_p08 = np.array([[-2.0, -1.0], [-0.5, 1.0], [0.75, -1.0], [2.0, 0.5]], dtype=np.float64)
t_p08 = np.array([-1.0, -1.0, 1.0, 1.0], dtype=np.float64)
w_p08 = np.array([0.6, -0.2], dtype=np.float64)
b_p08 = 0.1


def hinge_step(X, t, w, b, C, learning_rate):
    if not all(isinstance(a, np.ndarray) and a.dtype == np.float64 for a in (X, t, w)):
        raise ValueError("X, t, and w must be float64 arrays")
    if X.ndim != 2 or t.ndim != 1 or w.ndim != 1 or X.shape[0] < 1 or X.shape[1] < 1:
        raise ValueError("invalid dimensions")
    if t.shape != (X.shape[0],) or w.shape != (X.shape[1],):
        raise ValueError("shape mismatch")
    scalars = (b, C, learning_rate)
    if not all(np.isscalar(v) and np.isfinite(v) for v in scalars) or C <= 0 or learning_rate <= 0:
        raise ValueError("invalid scalars")
    if not all(np.isfinite(a).all() for a in (X, t, w)) or not np.all((t == -1.0) | (t == 1.0)):
        raise ValueError("invalid values")
    # PLAN018_MUTATION_TARGET: C12-p08-signed-hinge-branch
    margins = t * (X @ w + float(b))
    active = margins < 1.0
    objective = 0.5 * float(w @ w) + float(C) * float(np.maximum(0.0, 1.0 - margins).mean())
    grad_w = w - (float(C) / X.shape[0]) * (X[active].T @ t[active])
    grad_b = -(float(C) / X.shape[0]) * float(t[active].sum())
    return {"objective": float(objective), "margins": margins, "active": active,
            "grad_w": grad_w, "grad_b": float(grad_b),
            "w_next": w - float(learning_rate) * grad_w,
            "b_next": float(b) - float(learning_rate) * grad_b}


result_p08 = hinge_step(X_p08, t_p08, w_p08, b_p08, C=1.5, learning_rate=0.1)


### Answer check


In [ ]:
ATOL = 1e-12
RTOL = 1e-10
# PLAN018_ANSWER_CHECK: C12-p08-hinge-subgradient
assert np.allclose(result_p08["margins"], [0.9, 0.4, 0.75, 1.2], atol=ATOL, rtol=RTOL)
assert np.array_equal(result_p08["active"], [True, True, True, False])
assert np.isclose(result_p08["objective"], 0.55625, atol=ATOL, rtol=RTOL)
assert np.allclose(result_p08["grad_w"], [-0.61875, 0.175], atol=ATOL, rtol=RTOL)
assert np.isclose(result_p08["grad_b"], 0.375, atol=ATOL, rtol=RTOL)
assert np.allclose(result_p08["w_next"], [0.661875, -0.2175], atol=ATOL, rtol=RTOL)
assert np.isclose(result_p08["b_next"], 0.0625, atol=ATOL, rtol=RTOL)
